# MLB Game Simulator v1

**Keep `mlb_data.py`, `mlb_game_sim.py` and `mlb_daily.py` in the same folder as this notebook. If you replace a .py file, restart the kernel before running again.**

First run downloads about two seasons of boxscores (~5,000 games, 2-3 minutes) into `mlb_data/`. Later runs only fetch new games. No API key needed: DraftKings lines and props come through ESPN's public odds feed.

**How a game is simulated.** One plate appearance at a time, 6,000 times per game. Each PA's outcome comes from the batter's and pitcher's regressed platoon rates (odds-ratio method), adjusted for park, weather, umpire, home field, travel, defense, catcher framing and times through the order. Starters come out on a pitch count drawn from their own recent starts; bullpens are rebuilt every game from who is rested. Every market and player prop is read off the same simulated games, so same-game combos are internally consistent.

Requirements: `pip install pandas numpy requests openpyxl`

In [1]:
from mlb_daily import *
import pandas as pd
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)

## 1. Today's slate
Fits the model on every game before today (cached for 6 hours), checks the last 30 days' scoring environment, simulates every game, pulls DraftKings lines and props, logs every prediction, and writes `reports/mlb_<date>.xlsx` and `.html`.

In [2]:
report = daily_report()            # or daily_report("2026-10-06", sims=10000)

Loading platoon splits...
Fitted on 4945 games before 2026-10-28.
Recent environment (39 games, last 30 days): actual 7.18 runs/game, sim 7.77 before adjustment
  per-PA multipliers: K x1.000, BB x1.018, HBP x1.096, 1B x1.001, 2B x0.905, 3B x1.008, HR x0.929

=== MLB 2026-10-28 (1 games) ===
                              game start_az           away_sp home_sp  away_win  home_win  total  dk_total  nrfi
Milwaukee Brewers @ Tampa Bay Rays  5:00 PM Jacob Misiorowski bullpen      54.4      45.6   7.04       6.5  61.3

Model vs DraftKings, edge >= 3% (sorted by EV per $1):
                              game        market                    side  line  model_prob  dk_novig  dk_price  edge    ev
Milwaukee Brewers @ Tampa Bay Rays         walks    Richie Palacios over   0.5       0.351     0.236     297.0 0.115 0.395
Milwaukee Brewers @ Tampa Bay Rays          outs Jacob Misiorowski under  15.5       0.693     0.533    -133.0 0.160 0.215
Milwaukee Brewers @ Tampa Bay Rays         walks        

## 2. Betting board
`edge` = model probability minus DraftKings' no-vig probability. `ev` = expected profit per $1 at DraftKings' price.

In [3]:
best_bets(report).head(20)

,game,game_pk,market,side,line,model_prob,fair_odds,dk_price,dk_novig,edge,ev,dk_line_open,dk_price_open,push,type
44,Milwaukee Brewers @ Tampa Bay Rays,849799,walks,Richie Palacios over,0.5,0.351333,185,297.0,0.236062,0.115271,0.394793,0.5,291.0,0.0,prop
23,Milwaukee Brewers @ Tampa Bay Rays,849799,outs,Jacob Misiorowski under,15.5,0.693333,-226,-133.0,0.533066,0.160267,0.214637,16.5,-120.0,0.0,prop
55,Milwaukee Brewers @ Tampa Bay Rays,849799,walks,Yandy Díaz over,0.5,0.328667,204,260.0,0.259740,0.068926,0.183200,0.5,252.0,0.0,prop
25,Milwaukee Brewers @ Tampa Bay Rays,849799,walks_allowed,Jacob Misiorowski over,1.5,0.434833,130,158.0,0.363230,0.071603,0.121870,1.5,150.0,0.0,prop
24,Milwaukee Brewers @ Tampa Bay Rays,849799,strikeouts,Jacob Misiorowski under,5.5,0.545333,-120,-101.0,0.473170,0.072163,0.085267,5.5,-111.0,0.0,prop
47,Milwaukee Brewers @ Tampa Bay Rays,849799,walks,Taylor Walls over,0.5,0.289333,246,275.0,0.249721,0.039613,0.085000,0.5,275.0,0.0,prop
38,Milwaukee Brewers @ Tampa Bay Rays,849799,hrr,Liam Hicks over,0.5,0.674333,-207,-166.0,0.582968,0.091366,0.080558,0.5,-173.0,0.0,prop
28,Milwaukee Brewers @ Tampa Bay Rays,849799,singles,Jake Bauers over,0.5,0.441333,127,143.0,0.381679,0.059654,0.072440,0.5,152.0,0.0,prop
11,Milwaukee Brewers @ Tampa Bay Rays,849799,walks,Cooper Pratt over,0.5,0.333500,200,220.0,0.291480,0.042020,0.067200,0.5,207.0,0.0,prop
17,Milwaukee Brewers @ Tampa Bay Rays,849799,walks,Garrett Mitchell over,0.5,0.325667,207,227.0,0.285372,0.040294,0.064930,0.5,227.0,0.0,prop


In [4]:
report["board"]                    # every game market: moneyline, run line, total, team totals, F5, NRFI, first to score

,game,game_pk,market,side,line,model_prob,fair_odds,dk_price,dk_novig,edge,ev,dk_line_open,dk_price_open,push
0,Milwaukee Brewers @ Tampa Bay Rays,849799,moneyline,Milwaukee Brewers,NaN,0.544167,-119,-157.0,0.584210,-0.040043,-0.109230,NaN,-140.0,0.000000
1,Milwaukee Brewers @ Tampa Bay Rays,849799,moneyline,Tampa Bay Rays,NaN,0.455833,119,130.0,0.415790,0.040043,0.048417,NaN,116.0,0.000000
2,Milwaukee Brewers @ Tampa Bay Rays,849799,run line,Tampa Bay Rays,1.5,0.601833,-151,-142.0,0.561244,0.040589,0.025660,NaN,NaN,0.000000
3,Milwaukee Brewers @ Tampa Bay Rays,849799,run line,Milwaukee Brewers,-1.5,0.398167,151,118.0,0.438756,-0.040589,-0.131997,NaN,NaN,0.000000
4,Milwaukee Brewers @ Tampa Bay Rays,849799,total,over,6.5,0.513833,-106,-120.0,0.521739,-0.007906,-0.057972,6.5,-105.0,0.000000
5,Milwaukee Brewers @ Tampa Bay Rays,849799,total,under,6.5,0.486167,106,100.0,0.478261,0.007906,-0.027667,6.5,-115.0,0.000000
6,Milwaukee Brewers @ Tampa Bay Rays,849799,team total MIL,over,3.5,0.480833,108,-120.0,0.510121,-0.029288,-0.118472,NaN,NaN,0.000000
7,Milwaukee Brewers @ Tampa Bay Rays,849799,team total MIL,under,3.5,0.519167,-108,-110.0,0.489879,0.029288,-0.008864,NaN,NaN,0.000000
8,Milwaukee Brewers @ Tampa Bay Rays,849799,team total TB,over,2.5,0.552333,-123,-125.0,0.520305,0.032029,-0.005800,NaN,NaN,0.000000
9,Milwaukee Brewers @ Tampa Bay Rays,849799,team total TB,under,2.5,0.447667,123,-105.0,0.479695,-0.032029,-0.125984,NaN,NaN,0.000000


In [5]:
report["props"].sort_values("edge", ascending=False).head(25)   # player props vs DraftKings

,game,game_pk,player,pid,team,role,market,side,line,model_mean,model_prob,fair_odds,dk_price,dk_novig,edge,ev,push,dk_line_open,dk_price_open,dk_novig_open,model_prob_open,push_open,edge_open,ev_open
112,Milwaukee Brewers @ Tampa Bay Rays,849799,Jacob Misiorowski,694819,Milwaukee Brewers,pitcher,outs,under,15.5,13.828333,0.693333,-226,-133.0,0.533066,0.160267,0.214637,0.0,16.5,-120.0,0.509047,0.779667,0.0,0.270620,0.429389
205,Milwaukee Brewers @ Tampa Bay Rays,849799,Richie Palacios,680700,Tampa Bay Rays,batter,walks,over,0.5,0.416500,0.351333,185,297.0,0.236062,0.115271,0.394793,0.0,0.5,291.0,0.239598,0.351333,0.0,0.111735,0.373713
182,Milwaukee Brewers @ Tampa Bay Rays,849799,Liam Hicks,689414,Tampa Bay Rays,batter,hrr,over,0.5,1.602167,0.674333,-207,-166.0,0.582968,0.091366,0.080558,0.0,0.5,-173.0,0.593084,0.674333,0.0,0.081249,0.064121
137,Milwaukee Brewers @ Tampa Bay Rays,849799,Jonathan Aranda,666018,Tampa Bay Rays,batter,hrr,over,0.5,1.652000,0.674833,-208,-175.0,0.595142,0.079692,0.060452,0.0,0.5,-173.0,0.592032,0.674833,0.0,0.082801,0.064910
114,Milwaukee Brewers @ Tampa Bay Rays,849799,Jacob Misiorowski,694819,Milwaukee Brewers,pitcher,strikeouts,under,5.5,5.356833,0.545333,-120,-101.0,0.473170,0.072163,0.085267,0.0,5.5,-111.0,0.495845,0.545333,0.0,0.049489,0.036625
179,Milwaukee Brewers @ Tampa Bay Rays,849799,Liam Hicks,689414,Tampa Bay Rays,batter,hits,over,0.5,0.856333,0.606333,-154,-133.0,0.534307,0.072026,0.062223,0.0,0.5,-142.0,0.548457,0.606333,0.0,0.057876,0.033329
115,Milwaukee Brewers @ Tampa Bay Rays,849799,Jacob Misiorowski,694819,Milwaukee Brewers,pitcher,walks_allowed,over,1.5,1.454500,0.434833,130,158.0,0.363230,0.071603,0.121870,0.0,1.5,150.0,0.374611,0.434833,0.0,0.060222,0.087083
269,Milwaukee Brewers @ Tampa Bay Rays,849799,Yandy Díaz,650490,Tampa Bay Rays,batter,walks,over,0.5,0.388667,0.328667,204,260.0,0.259740,0.068926,0.183200,0.0,0.5,252.0,0.265516,0.328667,0.0,0.063151,0.156907
26,Milwaukee Brewers @ Tampa Bay Rays,849799,Chandler Simpson,802415,Tampa Bay Rays,batter,singles,over,0.5,0.810000,0.585833,-141,-129.0,0.521482,0.064351,0.039968,0.0,0.5,-124.0,0.512738,0.585833,0.0,0.073095,0.058280
134,Milwaukee Brewers @ Tampa Bay Rays,849799,Jonathan Aranda,666018,Tampa Bay Rays,batter,hits,over,0.5,0.864000,0.606167,-154,-138.0,0.541885,0.064282,0.045418,0.0,0.5,-140.0,0.544592,0.606167,0.0,0.061575,0.039143


In [6]:
report["player_projections"].sort_values("home_runs", ascending=False)     # mean projection for every lineup spot and starter

,side,team,player,pid,role,slot,PA,hits,total_bases,home_runs,runs,rbi,hrr,walks,singles,doubles,strikeouts,P(1+ hit),P(HR),game,outs,hits_allowed,walks_allowed,earned_runs,pitches,P(win)
12,home,Tampa Bay Rays,Junior Caminero,691406,batter,3,4.376333,0.899000,1.685333,0.215500,0.465500,0.521500,1.886000,0.376333,0.553667,0.119833,1.024667,0.623333,0.196333,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
0,away,Milwaukee Brewers,Jackson Chourio,694192,batter,1,4.871333,1.080667,1.795500,0.159833,0.532167,0.473167,2.086000,0.306667,0.701500,0.203333,1.238833,0.697167,0.148500,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
1,away,Milwaukee Brewers,Jake Bauers,641343,batter,2,4.752333,0.861667,1.481667,0.147333,0.515333,0.408167,1.785167,0.608000,0.554500,0.141667,1.540333,0.603667,0.139500,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
2,away,Milwaukee Brewers,Christian Yelich,592885,batter,3,4.641333,0.926500,1.512833,0.130500,0.453000,0.431000,1.810500,0.427167,0.616167,0.164833,1.334667,0.638167,0.123833,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
3,away,Milwaukee Brewers,William Contreras,661388,batter,4,4.530667,0.982500,1.523167,0.116000,0.457667,0.438667,1.878833,0.474167,0.682667,0.175000,0.854333,0.662000,0.109500,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
11,home,Tampa Bay Rays,Jonathan Aranda,666018,batter,2,4.475167,0.864000,1.353833,0.114500,0.421333,0.366667,1.652000,0.367833,0.609667,0.133333,1.474167,0.606167,0.110000,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
4,away,Milwaukee Brewers,Brice Turang,668930,batter,5,4.419833,0.941833,1.483500,0.106667,0.422333,0.440667,1.804833,0.433000,0.632000,0.184667,1.180833,0.643667,0.100167,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
16,home,Tampa Bay Rays,Ryan Vilade,668723,batter,7,3.956167,0.731167,1.164333,0.099667,0.336667,0.352500,1.420333,0.353500,0.510833,0.107167,1.093167,0.546500,0.095667,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
10,home,Tampa Bay Rays,Yandy Díaz,650490,batter,1,4.589667,1.036000,1.488167,0.097500,0.461333,0.344333,1.841667,0.388667,0.789333,0.138667,0.880833,0.680500,0.094500,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN
13,home,Tampa Bay Rays,Liam Hicks,689414,batter,4,4.279000,0.856333,1.285167,0.097167,0.376500,0.369333,1.602167,0.418000,0.634000,0.113000,0.699833,0.606333,0.092500,Milwaukee Brewers @ Tampa Bay Rays,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Explore one game
Change starters, lineups, scratches or weather and rerun with more simulations.

In [7]:
g = explore_game(report, "Dodgers")
# explore_game(report, "NYY", out=["Aaron Judge"], starters={"home": "Shane McClanahan"}, weather={"temp": 50, "wind_out": -8})

ValueError: No game for Dodgers

In [ ]:
# same-game combos priced from the same simulations
same_game(g, [("Cam Schlittler", "strikeouts", "over", 6.5), ("game", "total", "under", 6.5)])

Joint 12.1% (fair +729) vs 10.4% if the legs were independent


np.float64(0.1207)

## 4. Track record
Run the report before each slate on this computer so the log builds. After games finish, `grade_log()` settles every logged bet and compares the logged line with DraftKings' closing line (CLV).

In [ ]:
graded = grade_log()

Nothing graded yet (games not final).


## 5. Backtest
As-of-date: every game is simulated with a model fitted only on earlier games and graded against DraftKings' opening and closing lines. September 2026 takes about 15 minutes.

In [ ]:
bt = backtest("2026-10-01", "2026-10-04")
bt["summary"]["props"]

Loading platoon splits...
Fitted on 4914 games before 2026-10-01.
Recent environment (250 games, last 30 days): actual 9.16 runs/game, sim 9.19 before adjustment
  per-PA multipliers: K x1.036, BB x1.012, HBP x1.150, 1B x0.997, 2B x0.950, 3B x1.025, HR x0.972
  2026-10-01: 1 games simulated
  2026-10-03: 4 games simulated
  2026-10-04: 2 games simulated

================ BACKTEST vs DRAFTKINGS ================
Games with DraftKings lines: 7

winner:
  model log loss                     0.6628
  DK open log loss                   0.6484
  DK close log loss                  0.6577
  coin flip log loss                 0.6931
  model Brier                        0.2349
  DK close Brier                     0.2325
  model picks winner                 0.7143
  DK favorite wins                   0.7143
  50/50 blend log loss               0.6599
  model minus DK close log loss      0.0051
    90% range                        (-0.022028645211171462, 0.038314631105939606)

moneyline bets (close 

,market,offers,model MAE,model log loss,DK open log loss,bets,win%,ROI
0,doubles,125,0.206584,0.289818,0.298272,39,0.820513,0.071855
1,earned_runs,11,0.935303,0.787985,0.721628,8,0.250000,-0.527647
2,hits,125,0.651045,0.745254,0.715886,86,0.465116,-0.225116
3,hits_allowed,11,1.537515,0.839478,0.704376,8,0.375000,-0.275530
4,home_runs,125,0.208448,0.319179,0.322847,37,0.162162,0.165405
5,hrr,125,1.395715,0.706574,0.677667,86,0.360465,-0.329576
6,outs,11,3.253606,0.641747,0.721275,11,0.636364,0.238991
7,rbi,125,0.533312,0.508880,0.504807,37,0.378378,-0.223054
8,runs,125,0.492157,0.594643,0.584048,60,0.400000,-0.147947
9,singles,125,0.568179,0.674009,0.660535,61,0.327869,-0.285629


## 6. What the model is using
Fitted effects and assumptions for the current fit.

In [ ]:
report["model"].describe()

MLB model as of 2026-10-05 (season 2026)
  batters: 905  pitchers: 1310  parks: 37  umps: 95
  league per PA: K 0.222, BB 0.086, HBP 0.011, 1B 0.142, 2B 0.042, 3B 0.004, HR 0.031, OIP 0.463
  weather: HR +0.76%/degF, +0.77%/mph out; hits +0.05%/degF, +0.07%/mph out
  home batting multipliers: K 0.977, BB 1.023, HBP 1.008, 1B 1.008, 2B 1.014, 3B 1.063, HR 1.011, OIP 1.002
  long_trip: batting x0.986
  day_after_night: batting x0.994
  postseason starter leash: x0.898 (from 121 playoff starts)
  postseason environment (62 games): K x1.019, BB x0.998, HBP x1.117, 1B x0.980, 2B x0.955, 3B x1.002, HR x0.968
  recent-environment multipliers: K x1.030, BB x1.007, HBP x1.135, 1B x1.008, 2B x0.940, 3B x1.106, HR x1.004
